# Entropy, cross-entropy and KL divergence

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## Information is surprise
If I say "the sun will rise tomorrow" you learn nothing. If I say "it will snow in the desert tomorrow" you are very surprised. **The less probable an event, the more information it carries**:

$$I(x)=-\log_2 P(x)\quad\text{[bits]}$$

| Probability | Information (bits) | Example |
|---|---|---|
| 1 | 0 | A certain event |
| 0.5 | 1 | One coin flip |
| 0.25 | 2 | Two coins |
| 0.01 | 6.64 | A rare event |

## Entropy: the average surprise
$$H(p)=-\sum_k p_k\log_2 p_k$$

Entropy says how **uncertain** a distribution is.

| Distribution | Calculation | Entropy (bits) |
|---|---|---|
| Fair coin (0.5, 0.5) | $-2\times0.5\log_2 0.5$ | 1 |
| Biased coin (0.9, 0.1) | $-0.9\log_2 0.9-0.1\log_2 0.1$ | 0.469 |
| Two-headed coin (1, 0) | 0 | 0 |
| Fair die | $\log_2 6$ | 2.585 |

> [Left: the entropy of a coin peaks at $p=0.5$. Right: the less probability the model gives to the true class, the harsher the penalty.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_entropy.png)

*Left: the entropy of a coin peaks at $p=0.5$. Right: the less probability the model gives to the true class, the harsher the penalty.*

## Cross-entropy
If the true distribution is $p$ but the model predicts $q$:

$$H(p,q)=-\sum_k p_k\log q_k$$

Always $H(p,q)\ge H(p)$, with equality only when $q=p$. So minimizing the cross-entropy pulls q toward p.

In classification the true label is one-hot (for example $p=(1,0,0)$), so the formula reduces to **minus the log of the probability the model gave to the correct class**:

| Model prediction $q$ | Probability of the true class | Loss $-\ln q$ |
|---|---|---|
| (0.9, 0.05, 0.05) | 0.9 | 0.105 |
| (0.7, 0.2, 0.1) | 0.7 | 0.357 |
| (0.34, 0.33, 0.33) | 0.34 | 1.079 |
| (0.1, 0.2, 0.7) | 0.1 | 2.303 |

> A model that is confidently wrong is punished very hard: giving the true class a probability of 0.001 costs 6.9. This is what separates cross-entropy from "accuracy", which only counts right and wrong.

> Open the interactive lab for this part on the website: https://behrooztaheri.github.io/learn-python-ai/#/l/math-info

## KL divergence
The "extra cost" of using q in place of p:

$$D_{KL}(p\,\Vert\,q)=\sum_k p_k\log\frac{p_k}{q_k}=H(p,q)-H(p)\ \ge0$$

Example with $p=(0.5,0.5)$ and $q=(0.9,0.1)$:

| Quantity | Calculation | Value (bits) |
|---|---|---|
| $D_{KL}(p\Vert q)$ | $0.5\log_2\frac{0.5}{0.9}+0.5\log_2\frac{0.5}{0.1}$ | 0.737 |
| $D_{KL}(q\Vert p)$ | $0.9\log_2\frac{0.9}{0.5}+0.1\log_2\frac{0.1}{0.5}$ | 0.531 |

> KL is **not symmetric** ($0.737\ne0.531$), so it is not a "distance" in the mathematical sense.

## Where is it used?
| Where | What |
|---|---|
| Classification loss | Cross-entropy between the label and the softmax output |
| Decision trees | Information gain = the drop in entropy after a split |
| VAE | A KL term between the latent distribution and a standard normal |
| t-SNE | Minimizes the KL between similarities in two spaces |
| Knowledge distillation | KL between teacher and student outputs |
| Language models | perplexity $=2^{H}$ |

## Information gain in a decision tree
A node has 5 positive and 5 negative samples; its entropy is 1 bit. We split it into two branches (4 positive, 1 negative) and (1 positive, 4 negative):

| Node | Distribution | Entropy |
|---|---|---|
| Parent | (0.5, 0.5) | 1.000 |
| Each child | (0.8, 0.2) | 0.722 |
| Weighted average of the children | $0.5\times0.722+0.5\times0.722$ | 0.722 |
| Information gain | $1-0.722$ | 0.278 |



In [ ]:
import numpy as np

def entropy(p):
    p = np.asarray(p, dtype=float); p = p[p > 0]
    return -np.sum(p * np.log2(p))

def cross_entropy(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    return -np.sum(p[p > 0] * np.log2(q[p > 0]))

def kl(p, q):
    return cross_entropy(p, q) - entropy(p)

for name, p in [("fair coin", [0.5, 0.5]), ("biased coin", [0.9, 0.1]), ("certain", [1.0, 0.0]), ("fair die", [1 / 6] * 6)]:
    print(f"{name:12s} H = {entropy(p):.3f} bits")

# Cross-entropy loss for a one-hot label (natural log, as deep-learning libraries use)
for q in [[0.9, 0.05, 0.05], [0.7, 0.2, 0.1], [0.34, 0.33, 0.33], [0.1, 0.2, 0.7]]:
    print("q =", q, "  loss =", round(-np.log(q[0]), 3))

p, q = [0.5, 0.5], [0.9, 0.1]
print("KL(p||q) =", round(kl(p, q), 3), "   KL(q||p) =", round(kl(q, p), 3), "   KL(p||p) =", kl(p, p))

# Information gain of a split in a decision tree
parent = entropy([0.5, 0.5])
children = 0.5 * entropy([0.8, 0.2]) + 0.5 * entropy([0.2, 0.8])
print("information gain =", round(parent - children, 3))

# The same loss with scikit-learn
from sklearn.metrics import log_loss
y_true = [0, 0, 1, 2]
y_prob = [[0.9, 0.05, 0.05], [0.7, 0.2, 0.1], [0.2, 0.6, 0.2], [0.1, 0.2, 0.7]]
print("mean cross-entropy (sklearn log_loss):", round(log_loss(y_true, y_prob), 4))

### Exercise

What is the entropy of $(0.5, 0.25, 0.25)$ in bits? If the model predicts $(0.25, 0.5, 0.25)$, what are the cross-entropy and the KL divergence?

In [ ]:
# Your answer here

### Exercise

In the lab pick the one-hot label and push the weight of class A in q toward zero. What happens to the cross-entropy?

In [ ]:
# Your answer here